In [ ]:
import os

path = "./data" # 데이터를 생성해서 탑재하는 방식으로 진행.

# 입력한 값을 찾아서 하나의 변수로 설정
users = os.listdir(path)

is_target = False
target_user = '김부장' #input('이름을 입력해주세요: ')


for user in users:
    if target_user == user:
        is_target = True
        break
    else:
        is_target = False

if is_target:
    print(f"{target_user}의 일기를 찾았습니다.")
else:
    print(f"{target_user}의 일기를 찾을 수 없습니다.")

홍길동의 일기를 찾았습니다.


In [8]:
from glob import glob
from importlib.resources import path
import os
import numpy as np
from langchain_community.document_loaders import DirectoryLoader, JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["date"] = record.get("date")
    metadata["emotion"] = record.get("emotion")
    return metadata

loader = DirectoryLoader(
    path = f'./data/{target_user}',
    glob = '*.json',
    loader_cls=JSONLoader,
    loader_kwargs={
        'jq_schema' : '.[]',
        'content_key': 'diary',
        'metadata_func' : metadata_func
    }
)

docs = loader.load()

docs = sorted(docs, key=lambda x: x.metadata['date'], reverse=True)

print(len(docs))
print(docs)

7
[Document(metadata={'source': '/Users/siyoung/Archive-of-Feelings/data/홍길동/diary.json', 'seq_num': 7, 'date': '2026-03-18', 'emotion': '분노'}, page_content='아 점심시간 끝났다....\n이제 일해야되네 한숨만 나온다.\n왜 자꾸 일이 생기는 거지.\n나는 아무것도 안했는데 일을 시키네, 싸우자는 건가.'), Document(metadata={'source': '/Users/siyoung/Archive-of-Feelings/data/홍길동/diary.json', 'seq_num': 6, 'date': '2026-03-17', 'emotion': '분노'}, page_content='맨날 보는 사람들 정들법도한데 왜 이사람들은 매일매일 나를 화나게하는걸까요\n왜 서로 변하지 않는걸까요 그것은 내가 일을 못해서 그런거겠지 라고 생각하면서 오늘도 더 잘해보려고 노력해야지\n얼른 퇴근하고 강아지랑 놀고싶다. 침대에 누워서 자고 싶다.\n내일은 야근안하고 친구들만나서 놀고싶다. \n불금인데 왜 나만 야근을 하는거야 나도 친구 생일파티 가고싶다.'), Document(metadata={'source': '/Users/siyoung/Archive-of-Feelings/data/홍길동/diary.json', 'seq_num': 5, 'date': '2026-03-16', 'emotion': '불쾌함'}, page_content='일이 하기 싫다. 오늘은 늦게 일어나서 지각을 할 뻔했지만 지각을 하지 않았다. 다행이다.\n맨날 가기 싫다 가지말까 이고민하면서 지각하기 싫어서 열심히 씻고 준비하는 내가 너무 모순적이다.\n코로나 바이러스 때문에 외출을 삼가하세요 여행가지마세요 하면서 왜 출근은 하라는거지?\n출퇴근시간에 모든 사람을 다 만나느데 말이지'), Document(metadata={'source': '/Users/siyoung/Archive-of-

In [13]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
# True가 출력되어야 정상 
load_dotenv()

import os

# print(f"[API KEY]\n{os.environ['OPENAI_API_KEY'][:-20]}" + "*" * 20)

In [10]:
from langchain_openai import OpenAIEmbeddings

documents_to_embed = []
for doc in docs:
    rich_text = f"작성일: {doc.metadata['date']}, 감정상태: {doc.metadata['emotion']}, 일기내용: {doc.page_content}"
    doc.page_content = rich_text # 본문 업데이트
    documents_to_embed.append(doc.page_content)

# 4. 임베딩 모델 초기화 및 벡터화
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
document_vectors = embeddings_model.embed_documents(documents_to_embed)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


test_queries = [
    "요즘 시위 때문에 이동하기가 너무 짜증나고 화가 나.",          # 2026-03-13 (분노)
    "친구랑 다투고 치킨을 먹었어도 기분은 여전히 안 좋았어.",    # 2026-03-12 (분노)
    "쉬는 날이 끝나가는 게 너무 힘들어.",                          # 2026-03-15 (불쾌함)
    "우연히 오랜 친구를 만나서 너무 신기하고 기뻤어.",              # 2026-03-14 (놀라움)
    "야근 때문에 친구 생일 파티에 못 갔어.",                        # 2026-03-17 (분노)
    "출근하기 싫은데 지각은 하기 싫어서 어쩔 수 없이 일어났어.",    # 2026-03-16 (불쾌함)
]

# 그냥 자료를 한 번에 다 LLM에 제공을 하고, context window를 활용해서 => 심리 상담이 필요할 것 같아요...
# context window가 7일이라고 가정하면...,

print("\n--- 임베딩 검색 품질 검증 ---")
for query in test_queries:
    query_vector = embeddings_model.embed_query(query)
    similarities = cosine_similarity([query_vector], document_vectors)[0]
    
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]
    best_doc = docs[best_match_idx]
    
    print(f"\n[질문] {query}")
    print(f" 매칭된 일기 날짜: {best_doc.metadata['date']} (유사도: {best_score:.2f})") # 유사도를 좀 더 높이도록...
    print(f" 매칭된 감정: {best_doc.metadata['emotion']}")


--- 임베딩 검색 품질 검증 ---

[질문] 요즘 시위 때문에 이동하기가 너무 짜증나고 화가 나.
 -> 매칭된 일기 날짜: 2026-03-13 (유사도: 0.42)
 -> 매칭된 감정: 분노

[질문] 친구랑 다투고 치킨을 먹었어도 기분은 여전히 안 좋았어.
 -> 매칭된 일기 날짜: 2026-03-12 (유사도: 0.34)
 -> 매칭된 감정: 분노

[질문] 쉬는 날이 끝나가는 게 너무 힘들어.
 -> 매칭된 일기 날짜: 2026-03-15 (유사도: 0.50)
 -> 매칭된 감정: 불쾌함

[질문] 우연히 오랜 친구를 만나서 너무 신기하고 기뻤어.
 -> 매칭된 일기 날짜: 2026-03-14 (유사도: 0.37)
 -> 매칭된 감정: 놀라움

[질문] 야근 때문에 친구 생일 파티에 못 갔어.
 -> 매칭된 일기 날짜: 2026-03-17 (유사도: 0.39)
 -> 매칭된 감정: 분노

[질문] 출근하기 싫은데 지각은 하기 싫어서 어쩔 수 없이 일어났어.
 -> 매칭된 일기 날짜: 2026-03-16 (유사도: 0.46)
 -> 매칭된 감정: 불쾌함
